# LLMs As Re-Rankers

In [1]:
from glob import glob
from trectools import TrecRun, TrecQrel, TrecEval
from tqdm import tqdm
import pandas as pd

DATASET_TO_TREC_IDENTIFIER = {
    'trec-dl-2019-judged': 'trec28',
    'trec-dl-2020-judged': 'trec29',
}

MODEL_TO_NAME = {
    'AnthropicLLM-claude-3-haiku-20240307-umbrella_zeroshot_basic': 'Claude-3-haiku',
    'AnthropicLLM-claude-3-sonnet-20240229-umbrella_zeroshot_basic': 'Claude-3-sonnet',
    'LiteLLM-llama3-umbrella_zeroshot_basic': 'Llama-3',
    'LiteLLM-llama3.1-umbrella_zeroshot_basic': 'Llama-3.1',
    'GeminiGPT-gemini-1.5-flash-umbrella_zeroshot_basic': 'Gemini-1.5-flash',
    'GeminiGPT-gemini-1.5-flash-8b-umbrella_zeroshot_basic': 'Gemini-1.5-flash-8b',
    'OpenAiGPT-gpt-4o-umbrella_zeroshot_basic': 'GPT-4o',
    'OpenAiGPT-gpt-4o-mini-umbrella_zeroshot_basic': 'GPT-4o-mini',
}

RUNS = {}
QRELS = {}

def load_qrels(dataset, name):
    path = f'../data/msmarco-passage-{dataset}/qrels/*.qrels.txt'

    global qrels
    if dataset not in QRELS:
        QRELS[dataset] = {}
        for i in tqdm(glob(path), 'load qrels'):
            qrel_name = i.split('/')[-1].split('.qrels')[0]
            assert qrel_name and qrel_name not in QRELS[dataset]
            QRELS[dataset][qrel_name] = TrecQrel(i)
    
    ret = TrecQrel()
    ret.qrels_data = QRELS[dataset][name].qrels_data.copy()
    return ret

def load_runs(dataset):
    ret = {}

    path = f'../data/trec-system-runs/{DATASET_TO_TREC_IDENTIFIER[dataset]}/deep.passages/input.*.gz'

    global RUNS

    if dataset not in RUNS:
        topics = load_qrels(dataset, 'trec').topics()
        for i in tqdm(glob(path), 'load runs'):
            run_name = i.split('/')[-1].split('.')[1]
            assert run_name and run_name not in ret
            ret[run_name] = TrecRun(i)
            ret[run_name].run_data = ret[run_name].run_data[ret[run_name].run_data['query'].isin(topics)]
            
        RUNS[dataset] = ret
        ret = {}

    for run_name, run in RUNS[dataset].items():
        run_copy = TrecRun()
        run_copy.run_data = run.run_data.copy()
        ret[run_name] = run_copy

    return ret

def re_rank_with_llm(run, dataset, llm_for_re_ranking):
    qrels = load_qrels(dataset, llm_for_re_ranking)
    scores = {}
    for _, i in qrels.qrels_data.iterrows():
        if str(i['query']) not in scores:
            scores[str(i['query'])] = {}
        scores[str(i['query'])][str(i['docid'])] = int(i['rel'])

    ret = TrecRun()
    ret.run_data = run.run_data.copy()
    ret.run_data
    ret.run_data['score'] = ret.run_data.apply(lambda i: scores[str(i['query'])].get(str(i['docid']), -1), axis=1)

    trecformat = ret.run_data.sort_values(["query", "score", "docid"], ascending=[True,False,False]).reset_index()
    topX = trecformat.groupby("query")[["query","docid","score"]].head(1000)
    topX["rank"] = 1
    topX["rank"] = topX.groupby("query")["rank"].cumsum()
    ret.run_data = topX

    return ret

def evaluate(dataset, run_to_modify, llm_for_re_ranking, llm_for_eval):
    qrels = {
        'original': load_qrels(dataset, 'trec'),
        'llm_eval': load_qrels(dataset, llm_for_eval),
    }
    evals = []

    for run_name, run in load_runs(dataset).items():
        evals.append({'run': run_name, 'original': TrecEval(run=run, qrels=qrels['original']).get_ndcg(depth=10), 'llm': TrecEval(run=run, qrels=qrels['llm_eval']).get_ndcg(depth=10)})

        if run_name == run_to_modify:
            modified_run = re_rank_with_llm(run, dataset, llm_for_re_ranking)
            evals.append({'run': run_name + '-re-ranked', 'original': TrecEval(run=modified_run, qrels=qrels['original']).get_ndcg(depth=10), 'llm': TrecEval(run=modified_run, qrels=qrels['llm_eval']).get_ndcg(depth=10)})


    evals = pd.DataFrame(evals)

    evals = evals.sort_values(["original"], ascending=[False]).reset_index()
    evals["rank_original"] = 1
    evals["rank_original"] = evals["rank_original"].cumsum()

    evals = evals.sort_values(["llm"], ascending=[False]).reset_index()
    evals["llm_rank"] = 1
    evals["llm_rank"] = evals["llm_rank"].cumsum()

    re_rank_eval = evals[evals['run'] == run_to_modify + '-re-ranked']
    assert len(re_rank_eval) == 1
    re_rank_eval = re_rank_eval.iloc[0].to_dict()

    return {
        'run': run_to_modify,
        'dataset': dataset,
        'llm_for_re_ranking': llm_for_re_ranking,
        'llm_for_eval': llm_for_eval,
        'nDCG@10 Score (original)': re_rank_eval['original'],
        'nDCG@10 Rank (original)': re_rank_eval['rank_original'],
        'nDCG@10 Score (LLM)': re_rank_eval['llm'],
        'nDCG@10 Rank (LLM)': re_rank_eval['llm_rank'],
    }


In [2]:
for dataset in DATASET_TO_TREC_IDENTIFIER:
    load_runs(dataset)
    load_qrels(dataset, 'trec')


load runs: 100%|██████████| 59/59 [00:35<00:00,  1.67it/s]


In [ ]:
def all_test_permutations():
    for dataset in DATASET_TO_TREC_IDENTIFIER:
        for run in load_runs(dataset):
            for evaluation_model in MODEL_TO_NAME:
                for re_rank_model in MODEL_TO_NAME:
                    yield (dataset, run, evaluation_model, re_rank_model)

eval_results = []
for dataset, run, evaluation_model, re_rank_model in tqdm(list(all_test_permutations()), 'Create Re-Rank Evaluations'):
    eval_results.append(evaluate(dataset, run, evaluation_model, re_rank_model))

pd.DataFrame(eval_results).to_json('llm-re-ranker-evaluations.jsonl', lines=True, orient='records')


Create Re-Rank Evaluations:   1%|          | 49/6144 [01:42<3:32:10,  2.09s/it]